In [1]:
import pandas as pd

In [2]:
INPUT_FILE: str = "./resources/input/Conceptos PTESA.xlsx"
df_input: pd.DataFrame = pd.read_excel(INPUT_FILE)
df_input: pd.DataFrame = df_input['Concepto'].astype(str).str.split('|').explode().to_frame()
df_input.head()

,Concepto
0,COSTO DIRECTO DE OBRA
0,ADMINISTRACION (22%)
0,IMPROVISTOS (3%)
0,"UTILIDAD (4,2016806722%) MAS IVA 19%"
0,ESTUDIOS Y DISEÑOS MAS IVA 19%


In [3]:
import nltk
from nltk.corpus import stopwords
import re

nltk.download('stopwords')
STOP_WORDS = set(stopwords.words('spanish'))

def clean_concept(concept: str):
    if concept == "" or concept is None:
        return ""
    if (not isinstance(concept, str)):
        return str(concept)

    concept = concept.lower()
    tabla = str.maketrans(f"áäéëíïóöúü", "aaeeiioouu")
    concept = concept.translate(tabla)
    concept = re.sub(r'\(\s*[\d,.]+\s*%?\s*\)', '', concept)
    concept = re.sub(r'\b\d+\s*[xX]\s*\d+\b', '', concept)
    concept = re.sub(r'\b\d+(?:[.,]\d+)?\s*%', '', concept)
    meses: list[str] = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]
    pattern_meses = r'\b(?:' + '|'.join(meses) + r')\b'
    concept = re.sub(pattern_meses, 'mes', concept)
    concept = re.sub(r'\b\d+(?:[.,]\d+)?\b', '', concept)
    concept = re.sub(r'\s*[,.]\s*', ' ', concept)
    caracteres_especiales: str = '–-#()[]{}/:_*+.,~°";$&´='+"'"
    tabla = str.maketrans(caracteres_especiales, " " * len(caracteres_especiales))
    concept = concept.translate(tabla)
    concept = re.sub(r'\d+$', '', concept)
    concept = re.sub(r'[0-9]', ' ', concept)
    concept = re.sub(r'\s+', ' ', concept).strip()
    tokens = concept.split()
    tokens = [word for word in tokens if word not in STOP_WORDS and len(word) > 2]
    return " ".join(tokens)

clean_concept("Patata")

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


'patata'

In [4]:
df_concepto_cleared: pd.DataFrame = df_input.copy()
df_concepto_cleared["concepto_cleared"] = df_concepto_cleared["Concepto"].apply(clean_concept)
df_concepto_cleared.head()

,Concepto,concepto_cleared
0,COSTO DIRECTO DE OBRA,costo directo obra
0,ADMINISTRACION (22%),administracion
0,IMPROVISTOS (3%),improvistos
0,"UTILIDAD (4,2016806722%) MAS IVA 19%",utilidad mas iva
0,ESTUDIOS Y DISEÑOS MAS IVA 19%,estudios diseños mas iva


In [5]:
df_clear: pd.DataFrame = df_concepto_cleared[["concepto_cleared"]].drop_duplicates()
print(f"Cantidad original: {len(df_concepto_cleared):,.0f}")
print(f"Cantidad limpia: {len(df_clear):,.0f} => reducción del {100*(1 - len(df_clear)/len(df_concepto_cleared)):.2f}%")
df_clear.head()

Cantidad original: 131,337
Cantidad limpia: 23,115 => reducción del 82.40%


,concepto_cleared
0,costo directo obra
0,administracion
0,improvistos
0,utilidad mas iva
0,estudios diseños mas iva


In [6]:
INPUT_HOMOLOGA: str = "./resources/input/Tabla_homologación_2.xlsx"
df_homologa: pd.DataFrame = pd.read_excel(INPUT_HOMOLOGA)
df_homologa[~df_homologa["KeyWords"].isna()].head()

,Concepto,KeyWords
17,Honorarios y Comisiones,COMISION POR ADMINISTRACION
24,Contratos de construcción y urbanización,"COSTO DIRECTO, IMPREVISTOS, COSTOS DIRECTOS"
32,Cuota de admon,"CUOTA AIRE ACONDICIONADO, CUOTA EXPENSAS COMUN..."


In [7]:
df_keywords: pd.DataFrame = df_homologa[~df_homologa["KeyWords"].isna()].copy()
dict_keyword_concept: dict[str, str] = {}
for ix, row in df_keywords.iterrows():
    cleared_keywords: list[str] = [clean_concept(kw) for kw in row["KeyWords"].split(",") if len(clean_concept(kw)) > 0]
    for kw in cleared_keywords:
        dict_keyword_concept[kw] = row["Concepto"]
dict_keyword_concept

{'comision administracion': 'Honorarios y Comisiones',
 'costo directo': 'Contratos de construcción y urbanización',
 'imprevistos': 'Contratos de construcción y urbanización',
 'costos directos': 'Contratos de construcción y urbanización',
 'cuota aire acondicionado': 'Cuota de admon',
 'cuota expensas comunes': 'Cuota de admon',
 'cuota publicidad': 'Cuota de admon',
 'modulo aire acondicionado': 'Cuota de admon',
 'modulo gastos generales': 'Cuota de admon',
 'modulo publicidad': 'Cuota de admon',
 'expensas comunes': 'Cuota de admon',
 'modulo mercadeo publicidad': 'Cuota de admon'}

In [8]:
INPUT_HOMOLOGA: str = "./resources/input/Tabla_homologación_2.xlsx"
df_explicacion_tributario: pd.DataFrame = pd.read_excel(INPUT_HOMOLOGA, sheet_name="Hoja 4")
df_explicacion_tributario[["Concepto", "Descripción"]].head()

,Concepto,Descripción
0,Compras Generales,"Adquisición de bienes tangibles, materiales e ..."
1,Productos agrícolas/pecuarios (sin proceso ind...,Compra o suministro de productos del sector ag...
2,Café Pergamino o Cereza,Adquisición de café en estado de cereza (recié...
3,Combustibles derivados del petróleo,Compra o suministro de combustibles líquidos o...
4,Vehículos (Adquisición),"Compra de automotores, maquinaria rodante y de..."


In [9]:
from pandas._libs.missing import NAType
from typing import Union

# Categoriza por KeyWord.
def fast_category(concept: str) -> Union[str, NAType]:
    if len(concept) <= 3:
        return "CONCEPTO VACIO"
    for kw in dict_keyword_concept:
        if kw == concept or kw in concept:
            return dict_keyword_concept[kw]
    return pd.NA

df_categorized: pd.DataFrame = df_clear.copy()
df_categorized["categoria"] = df_categorized["concepto_cleared"].apply(fast_category)
df_categorized[~df_categorized["categoria"].isna()].head()

,concepto_cleared,categoria
0,costo directo obra,Contratos de construcción y urbanización
30,costo directo,Contratos de construcción y urbanización
30,imprevistos,Contratos de construcción y urbanización
32,,CONCEPTO VACIO
89,costo directo acta parcial playa rica sede pri...,Contratos de construcción y urbanización


In [10]:
df_process: pd.DataFrame = df_categorized.copy()
print(f"Cantidad de conceptos: {len(df_process):,.0f}, cantidad sin categorizar: {len(df_process[~df_process['categoria'].isna()]):,.0f}, faltante: {len(df_process) - len(df_process[~df_process['categoria'].isna()]):,.0f}")
print(f"Se esperan un total de {len(df_process)/20:,.0f} batchs")

Cantidad de conceptos: 23,115, cantidad sin categorizar: 274, faltante: 22,841
Se esperan un total de 1,156 batchs


In [11]:
from typing import List

CONCEPTOS_TRIBUTARIOS: List[str] = [f"{row.Concepto}: {row.Descripción}" for ix, row in df_explicacion_tributario[["Concepto", "Descripción"]].iterrows()]
CONCEPTOS_TRIBUTARIOS.append("DESCONOCIDO - No se logra entender o definir el concepto al que pertenece")

In [12]:
import os
import json
from typing import TypedDict, Optional
from typing_extensions import TypedDict
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

client: genai.Client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
MODEL: str = "gemini-3.1-flash-lite-preview"

SYSTEM_PROMPT: str = f"""Eres un experto en tributación colombiana especializado en retención en la fuente y clasificación de conceptos de facturación.

Tu única tarea es recibir la descripción de un concepto de factura y clasificarlo en exactamente uno de los siguientes conceptos tributarios:

{chr(10).join(f"- {c}" for c in CONCEPTOS_TRIBUTARIOS)}

Reglas de clasificación:
- "Empresas de Servicios Temporales" aplica cuando el concepto describe suministro, provisión o gestión de personal temporal, nómina externa o mano de obra en misión.
- "Honorarios y Comisiones" aplica a personas naturales o jurídicas que prestan servicios profesionales independientes sin relación laboral.
- "Consultoría General/Administración Delegada (PJ)" aplica cuando una persona jurídica presta asesoría, consultoría o administración de procesos.
- "Servicios Generales" es el comodín para servicios que no encajan claramente en ninguna categoría específica.
- "DEFAULT_DESCONOCIDO" solo se usa cuando el concepto es completamente ambiguo o no corresponde a ninguna categoría.

Niveles de confianza:
- TOTAL: El concepto no deja ninguna duda. Hay palabras clave directas.
- ALTA: Muy probablemente correcto, pero podría depender del proveedor o contexto del contrato.
- MEDIA: Confuso entre dos o más categorías. Se necesitaría más contexto para estar seguro.
- BAJA: No se pudo definir con claridad en ninguna categoría conocida.

Responde SIEMPRE en el formato JSON indicado, sin texto adicional.
"""


# ── Tipos ────────────────────────────────────────────────────────────────────

class ClasificacionTributaria(TypedDict):
    concepto: str      # Uno de los valores de CONCEPTOS_TRIBUTARIOS
    confianza: str     # "TOTAL" | "ALTA" | "MEDIA" | "BAJA"
    explicacion: str   # Justificación breve


# ── Caché del system prompt ──────────────────────────────────────────────────

def crear_cache(ttl: str = "3600s") -> str:
    """
    Sube el system prompt a la API de Gemini como contenido cacheado.
    Devuelve el nombre del caché para reutilizarlo en cada llamada.

    Nota: Gemini exige un mínimo de tokens para cachear (≥1 024).
    Si el prompt es muy corto, la API lanzará un error.
    """
    cached = client.caches.create(
        model=MODEL,
        config=types.CreateCachedContentConfig(
            system_instruction=SYSTEM_PROMPT,
            ttl=ttl,
        ),
    )
    print(f"[caché creado] nombre={cached.name!r}  expira={cached.expire_time}")
    return cached.name


# ── Clasificación ────────────────────────────────────────────────────────────

def categorizar_concepto_llm(
    concepto: str,
    cache_name: str,
    *,
    stream: bool = False,
) -> ClasificacionTributaria:
    """
    Clasifica un concepto de factura.

    Args:
        concepto:   Texto del concepto a clasificar.
        cache_name: Nombre del CachedContent con el system prompt.
        stream:     Si True, imprime los chunks a medida que llegan
                    (útil para depurar) y luego devuelve el objeto final.
    """
    contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=concepto)])
    ]

    config = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        cached_content=cache_name,          # ← reutiliza el system prompt cacheado
        response_mime_type="application/json",
        response_schema=ClasificacionTributaria,  # ← fuerza la estructura JSON
    )

    if stream:
        chunks: list[str] = []
        for chunk in client.models.generate_content_stream(
            model=MODEL, contents=contents, config=config
        ):
            print(chunk.text, end="", flush=True)
            chunks.append(chunk.text)
        print()  # salto de línea final
        raw = "".join(chunks)
    else:
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=config
        )
        raw = response.text

    return json.loads(raw)

In [13]:
cache_name = crear_cache(ttl="3600s")   # crear una sola vez; reutilizar después
conceptos_prueba = [
    "costo directo obra",
    "suministro de personal temporal para planta",
    "asesoría jurídica tributaria",
]

for concepto in conceptos_prueba:
    resultado: ClasificacionTributaria = categorizar_concepto_llm(concepto, cache_name)
    print(f"\nConcepto: {concepto!r}")
    print(f"  → {resultado['concepto']} [{resultado['confianza']}]")
    print(f"     {resultado['explicacion']}")

[caché creado] nombre='cachedContents/l3gbad07qen700k24qzm0c69kyh9md1997tpkysd'  expira=2026-03-10 15:11:24.494945+00:00

Concepto: 'costo directo obra'
  → Contratos de construcción y urbanización [ALTA]
     El término 'costo directo de obra' es una nomenclatura estándar utilizada en la contabilidad y gestión de contratos de construcción y obra civil. Refiere a la ejecución material de proyectos, por lo cual se clasifica dentro de los contratos de construcción y urbanización.

Concepto: 'suministro de personal temporal para planta'
  → Empresas de Servicios Temporales [TOTAL]
     El concepto describe explícitamente el 'suministro de personal temporal', lo cual encaja perfectamente con la definición de actividades realizadas por Empresas de Servicios Temporales (EST) según la normativa y las reglas de clasificación proporcionadas.

Concepto: 'asesoría jurídica tributaria'
  → Honorarios y Comisiones [TOTAL]
     La asesoría jurídica y tributaria se clasifica como un servicio intelect

In [14]:

import os
import json
import time
import logging
from datetime import datetime, timezone
from typing import List
from typing_extensions import TypedDict
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

client: genai.Client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
MODEL: str = "gemini-3.1-flash-lite-preview"

BATCH_SIZE: int = 20
CACHE_TTL: str = "3600s"          # 1 h; la caché se renueva automáticamente si expira
CACHE_SAFETY_MARGIN: int = 120    # segundos antes del vencimiento para renovar


# ── Tipos ────────────────────────────────────────────────────────────────────

class ClasificacionItem(TypedDict):
    concepto: str      # Concepto original (espejo del input para verificar integridad)
    categoria: str     # Uno de los valores de CONCEPTOS_TRIBUTARIOS
    explicacion: str   # Justificación breve (máx ~40 palabras)


def build_system_prompt(conceptos_tributarios: List[str]) -> str:
    opciones = "\n".join(f"- {c}" for c in conceptos_tributarios)
    return f"""Eres un experto en tributación colombiana especializado en retención en la fuente y clasificación de conceptos de facturación.

Tu única tarea es recibir una lista numerada de conceptos de factura y clasificar cada uno en exactamente uno de los siguientes conceptos tributarios:

{opciones}

Reglas de clasificación:
- "Empresas de Servicios Temporales" aplica cuando el concepto describe suministro, provisión o gestión de personal temporal, nómina externa o mano de obra en misión.
- "Honorarios y Comisiones" aplica a personas naturales o jurídicas que prestan servicios profesionales independientes sin relación laboral.
- "Consultoría General/Administración Delegada (PJ)" aplica cuando una persona jurídica presta asesoría, consultoría o administración de procesos.
- "Servicios Generales" es el comodín para servicios que no encajan claramente en ninguna categoría específica.
- "DESCONOCIDO" solo se usa cuando el concepto es completamente ambiguo o no corresponde a ninguna categoría.

Devuelve SIEMPRE un array JSON con exactamente tantos objetos como conceptos recibiste, en el MISMO ORDEN, con esta estructura:
[
  {{
    "concepto": "<texto original del concepto tal como fue recibido>",
    "categoria": "<categoría clasificada>",
    "explicacion": "<justificación en máximo 40 palabras>"
  }},
  ...
]

Sin texto adicional fuera del JSON.
"""

In [15]:

# ── Gestión de caché con auto-renovación ─────────────────────────────────────

class CacheManager:
    """Mantiene una caché de system prompt activa y la renueva antes de que expire."""

    def __init__(self, conceptos_tributarios: List[str]):
        self._conceptos = conceptos_tributarios
        self._name: str | None = None
        self._expire_time: datetime | None = None

    @property
    def _is_valid(self) -> bool:
        if not self._name or not self._expire_time:
            return False
        remaining = (self._expire_time - datetime.now(timezone.utc)).total_seconds()
        return remaining > CACHE_SAFETY_MARGIN

    def get(self) -> str:
        """Devuelve un nombre de caché válido, creando o renovando si es necesario."""
        if not self._is_valid:
            self._create()
        return self._name  # type: ignore[return-value]

    def invalidate(self) -> str:
        """Fuerza la recreación de la caché (llamar si la API reporta caché inválida)."""
        log.warning("Caché inválida según la API; recreando…")
        self._name = None
        return self.get()

    def _create(self) -> None:
        prompt = build_system_prompt(self._conceptos)
        cached = client.caches.create(
            model=MODEL,
            config=types.CreateCachedContentConfig(
                system_instruction=prompt,
                ttl=CACHE_TTL,
            ),
        )
        self._name = cached.name
        self._expire_time = cached.expire_time
        log.info("Caché creada: name=%r  expira=%s", self._name, self._expire_time)


# ── Llamada al modelo para un batch ──────────────────────────────────────────

def _call_api(batch: List[str], cache_name: str) -> List[ClasificacionItem]:
    """Clasifica un batch de conceptos en una sola llamada."""
    numbered = "\n".join(f"{i+1}. {c}" for i, c in enumerate(batch))
    contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=numbered)])
    ]
    config = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        cached_content=cache_name,
        response_mime_type="application/json",
        response_schema=list[ClasificacionItem],
    )
    response = client.models.generate_content(model=MODEL, contents=contents, config=config)
    return json.loads(response.text)


def categorizar_batch(
    batch: List[str],
    cache_mgr: CacheManager,
    *,
    retries: int = 3,
) -> List[ClasificacionItem]:
    """
    Clasifica un batch de hasta BATCH_SIZE conceptos con reintentos ante errores de caché.

    Args:
        batch:      Lista de textos de conceptos (máx BATCH_SIZE).
        cache_mgr:  Instancia de CacheManager para obtener/renovar la caché.
        retries:    Número máximo de reintentos ante fallos recuperables.
    """
    for attempt in range(1, retries + 1):
        try:
            cache_name = cache_mgr.get()
            resultado = _call_api(batch, cache_name)

            # Verificación de integridad: mismo número de items y orden preservado
            if len(resultado) != len(batch):
                raise ValueError(
                    f"El modelo devolvió {len(resultado)} items para un batch de {len(batch)}"
                )
            for orig, item in zip(batch, resultado):
                if item["concepto"].strip() != orig.strip():
                    log.warning(
                        "Concepto devuelto %r no coincide con original %r; corrigiendo.",
                        item["concepto"], orig,
                    )
                    item["concepto"] = orig   # forzar el original como fuente de verdad

            return resultado

        except Exception as exc:
            msg = str(exc).lower()
            is_cache_error = "cache" in msg or "not found" in msg or "expired" in msg
            if is_cache_error and attempt < retries:
                cache_mgr.invalidate()
                continue
            if attempt < retries:
                wait = 2 ** attempt
                log.warning("Intento %d/%d falló (%s); reintentando en %ds…", attempt, retries, exc, wait)
                time.sleep(wait)
            else:
                log.error("Batch falló tras %d intentos: %s", retries, exc)
                raise


# ── Procesamiento completo de una lista larga ─────────────────────────────────

def clasificar_conceptos(
    conceptos: List[str],
    conceptos_tributarios: List[str],
) -> List[ClasificacionItem]:
    """
    Clasifica una lista de conceptos de factura en batches de BATCH_SIZE.

    Args:
        conceptos:             Lista completa de textos a clasificar.
        conceptos_tributarios: Lista de categorías válidas (incluida "DESCONOCIDO").

    Returns:
        Lista de ClasificacionItem en el mismo orden que la entrada.
    """
    cache_mgr = CacheManager(conceptos_tributarios)
    resultados: List[ClasificacionItem] = []

    total = len(conceptos)
    n_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(n_batches):
        batch = conceptos[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
        log.info("Procesando batch %d/%d (%d conceptos)…", i + 1, n_batches, len(batch))
        parcial = categorizar_batch(batch, cache_mgr)
        resultados.extend(parcial)
        log.info("Batch %d/%d completado. Total procesados: %d/%d", i + 1, n_batches, len(resultados), total)

    return resultados


In [16]:
conceptos_prueba = [
    "costo directo obra",
    "suministro de personal temporal para planta",
    "asesoría jurídica tributaria",
    # … agregar más según sea necesario
]

resultados = clasificar_conceptos(conceptos_prueba, CONCEPTOS_TRIBUTARIOS)

for r in resultados:
    print(f"\nConcepto : {r['concepto']!r}")
    print(f"Categoría: {r['categoria']}")
    print(f"Explica  : {r['explicacion']}")

2026-03-10 14:11:44,617 INFO Procesando batch 1/1 (3 conceptos)…
2026-03-10 14:11:45,769 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/cachedContents "HTTP/1.1 200 OK"
2026-03-10 14:11:45,771 INFO Caché creada: name='cachedContents/sgid468rnlr2h0m8uf20fw27uwmppzcjvn0h0f9c'  expira=2026-03-10 15:11:44.973898+00:00
2026-03-10 14:11:45,771 INFO AFC is enabled with max remote calls: 10.
2026-03-10 14:11:52,031 INFO HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite-preview:generateContent "HTTP/1.1 200 OK"
2026-03-10 14:11:52,033 WARNING Concepto devuelto '1. costo directo obra' no coincide con original 'costo directo obra'; corrigiendo.
2026-03-10 14:11:52,034 WARNING Concepto devuelto '2. suministro de personal temporal para planta' no coincide con original 'suministro de personal temporal para planta'; corrigiendo.
2026-03-10 14:11:52,034 WARNING Concepto devuelto '3. asesoría jurídica tributaria' no coincide con o


Concepto : 'costo directo obra'
Categoría: Contratos de construcción y urbanización
Explica  : Los costos directos asociados a una obra se derivan de la ejecución de contratos de construcción, remodelación o urbanización de bienes inmuebles, según la normativa técnica y contable aplicable.

Concepto : 'suministro de personal temporal para planta'
Categoría: Empresas de Servicios Temporales
Explica  : El concepto describe explícitamente el suministro de mano de obra en misión para cubrir necesidades operativas, lo cual encaja directamente en la definición de empresas de servicios temporales.

Concepto : 'asesoría jurídica tributaria'
Categoría: Honorarios y Comisiones
Explica  : La asesoría profesional en temas jurídicos y tributarios constituye una prestación de servicios intelectuales de alto valor, realizada sin subordinación laboral, clasificándose como honorarios.
